# AUDIO/VISUAL SPEECH RECOGNITION

**Note** This tutorial requires `mediapipe` or `retinaface` detector. Please refer to [preparation](../preparation#setup) for installation.

**Note** To run this tutorial, please make sure you are in tutorials folder.

In [21]:
import sys
sys.path.insert(0, "../")

In [22]:
import os
import torch
import torchaudio
import torchvision

## 1. Build an inference pipeline

The InferencePipeline carries out the following three steps:

1. Load audio or video data
2. Run pre-processing functions
3. Run inference

In [23]:
import os
from lightning import ModelModule
from datamodule.transforms import AudioTransform, VideoTransform

In [24]:
import argparse
parser = argparse.ArgumentParser()
args, _ = parser.parse_known_args(args=[])

In [35]:
class InferencePipeline(torch.nn.Module):
    def __init__(self, args, ckpt_path, detector="mediapipe"):
        super(InferencePipeline, self).__init__()
        self.modality = args.modality
        if self.modality == "audio":
            self.audio_transform = AudioTransform(subset="test")
        elif self.modality == "video":
            if detector == "mediapipe":
                from preparation.detectors.mediapipe.detector import LandmarksDetector
                from preparation.detectors.mediapipe.video_process import VideoProcess
                self.landmarks_detector = LandmarksDetector()
                self.video_process = VideoProcess(convert_gray=False)
            elif detector == "retinaface":
                from preparation.detectors.retinaface.detector import LandmarksDetector
                from preparation.detectors.retinaface.video_process import VideoProcess
                self.landmarks_detector = LandmarksDetector(device="cuda:0")
                self.video_process = VideoProcess(convert_gray=False)
            self.video_transform = VideoTransform(subset="test")

        ckpt = torch.load(ckpt_path, map_location=lambda storage, loc: storage)
        self.modelmodule = ModelModule(args)
        self.modelmodule.model.load_state_dict(ckpt)
        self.modelmodule.eval()

    def load_video(self, data_filename):
        return torchvision.io.read_video(data_filename, pts_unit="sec")[0].numpy()

    def forward(self, data_filename):
        data_filename = os.path.abspath(data_filename)
        assert os.path.isfile(data_filename), f"data_filename: {data_filename} does not exist."

        if self.modality == "audio":
            audio, sample_rate = self.load_audio(data_filename)
            audio = self.audio_process(audio, sample_rate)
            audio = audio.transpose(1, 0)
            audio = self.audio_transform(audio)
            with torch.no_grad():
                transcript = self.modelmodule(audio)

        if self.modality == "video":
            video = self.load_video(data_filename)
            landmarks = self.landmarks_detector(video)
            video = self.video_process(video, landmarks)
            video = torch.tensor(video)
            print(video.shape)
            video = video.permute((0, 3, 1, 2))
            print(video.shape)
            video = self.video_transform(video)
            print(video.shape)
            with torch.no_grad():
                transcript = self.modelmodule(video)

        return transcript

    def load_audio(self, data_filename):
        waveform, sample_rate = torchaudio.load(data_filename, normalize=True)
        return waveform, sample_rate

    def load_video(self, data_filename):
        return torchvision.io.read_video(data_filename, pts_unit="sec")[0].numpy()

    def audio_process(self, waveform, sample_rate, target_sample_rate=16000):
        if sample_rate != target_sample_rate:
            waveform = torchaudio.functional.resample(
                waveform, sample_rate, target_sample_rate
            )
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        return waveform

## 2. Download a video from web

In [26]:
!wget --content-disposition https://media.spreadthesign.com/video/mp4/13/58463.mp4 -O ./input.mp4
data_filename = "input.mp4"

--2025-02-13 12:32:40--  https://media.spreadthesign.com/video/mp4/13/58463.mp4
Resolving media.spreadthesign.com (media.spreadthesign.com)... 13.224.222.72, 13.224.222.92, 13.224.222.15, ...
Connecting to media.spreadthesign.com (media.spreadthesign.com)|13.224.222.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 101130 (99K) [video/mp4]
Saving to: ‘./input.mp4’

./input.mp4         100%[===================>]  98.76K  --.-KB/s    in 0.009s  

2025-02-13 12:32:41 (10.9 MB/s) - ‘./input.mp4’ saved [101130/101130]



## 3. VSR inference

### 3.1 Download a pre-trained model

In [27]:
# !wget http://www.doc.ic.ac.uk/~pm4115/autoAVSR/vsr_trlrs3_base.pth -O ./vsr_trlrs3_base.pth
model_path = "./vsr_trlrs3_base.pth"

### 3.2 Initialize VSR pipeline

In [36]:
model_path

'./asr_trlrs3_base.pth'

In [37]:
setattr(args, 'modality', 'video')
pipeline = InferencePipeline(args, model_path, detector="mediapipe")

I0000 00:00:1739450161.919665  913080 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1739450161.948666 1089095 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.127.08), renderer: Tesla P40/PCIe/SSE2
W0000 00:00:1739450161.954874 1089090 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1739450161.961894  913080 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1739450161.979848 1089166 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.127.08), renderer: Tesla P40/PCIe/SSE2
W0000 00:00:1739450162.025359 1089161 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


EOFError: 

### 3.3 Run inference

In [38]:
# transcript = pipeline("input.mp4")
transcript = pipeline("/users/zifan/BOBSL/derivatives/islr_videos/test/mouthing/american/5970049209380884362-2286_881.mp4")
print(transcript)

torch.Size([21, 96, 96, 3])
torch.Size([21, 3, 96, 96])
torch.Size([21, 1, 88, 88])
AMERICA


## 4. ASR inference

### 4.1 Download a pre-trained model

In [31]:
# !wget http://www.doc.ic.ac.uk/~pm4115/autoAVSR/asr_trlrs3_base.pth -O ./asr_trlrs3_base.pth
model_path = "./asr_trlrs3_base.pth"

--2025-02-13 12:32:48--  http://www.doc.ic.ac.uk/~pm4115/autoAVSR/asr_trlrs3_base.pth
Resolving www.doc.ic.ac.uk (www.doc.ic.ac.uk)... 146.169.13.6
Connecting to www.doc.ic.ac.uk (www.doc.ic.ac.uk)|146.169.13.6|:80... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-02-13 12:32:48 ERROR 404: Not Found.



### 4.2 Initialize ASR pipeline

In [33]:
setattr(args, 'modality', 'audio')
pipeline = InferencePipeline(args, model_path)

EOFError: 

### 4.3 Run inference

In [ ]:
transcript = pipeline("input.mp4")
print(transcript)